# 05 — Evaluation & Model Comparison
**BraTS 2023 GLI Brain Tumor Segmentation | RCOEM 2026–27**

This notebook:
1. Loads best checkpoints of all trained models
2. Runs sliding window inference on the held-out test set
3. Computes all 5 metrics × 3 sub-regions = **15 numbers per model**
4. Generates the **final comparison table** for the project report
5. Produces bar charts and box plots for visualisation

**Estimated time:** ~30 minutes

In [ ]:
import sys, os
DATASET_PATH = '/home/yourname/BraTS2023_Training_Data'  # ← CHANGE THIS
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from tqdm.notebook import tqdm
from pathlib import Path

from src.models  import UNet3D, AttentionUNet3D
from src.dataset import BraTS2023InferenceDataset, get_patient_folders
from src.metrics import compute_patient_metrics, aggregate_metrics, print_metrics_table
from src.utils   import load_checkpoint, sliding_window_inference
from src.config  import DEVICE, PATCH_SIZE, PATCH_OVERLAP, N_FOLDS, MAX_PATIENTS, RANDOM_SEED

print(f'Device: {DEVICE}')
print('✅ All imports OK')

## 1️⃣ Define Models to Evaluate

In [ ]:
# All models to evaluate — extend this list as you train more variants
MODEL_CONFIGS = [
    {
        'name':       'Baseline 3D U-Net',
        'model_name': 'baseline_unet3d',
        'model':      UNet3D(in_channels=4, out_channels=4, init_features=32),
    },
    {
        'name':       'Attention U-Net (Channel only)',
        'model_name': 'attention_unet3d_channel_only',
        'model':      AttentionUNet3D(in_channels=4, out_channels=4, init_features=32,
                                      use_channel_attn=True, use_spatial_attn=False),
    },
    {
        'name':       'Attention U-Net (Spatial only)',
        'model_name': 'attention_unet3d_spatial_only',
        'model':      AttentionUNet3D(in_channels=4, out_channels=4, init_features=32,
                                      use_channel_attn=False, use_spatial_attn=True),
    },
    {
        'name':       'Attention U-Net (Full — Proposed)',
        'model_name': 'attention_unet3d_full',
        'model':      AttentionUNet3D(in_channels=4, out_channels=4, init_features=32,
                                      use_channel_attn=True, use_spatial_attn=True),
    },
]

# Load best checkpoints for each model
for cfg in MODEL_CONFIGS:
    cfg['model'] = cfg['model'].to(DEVICE)
    start, best_dice = load_checkpoint(cfg['model'], optimizer=None,
                                       model_name=cfg['model_name'], prefer_best=True)
    cfg['model'].eval()
    print(f"  Loaded: {cfg['name']:45s} | Best Val Dice: {best_dice:.4f}")

## 2️⃣ Get Test Patient Folders

In [ ]:
from src.dataset import get_subject_splits
all_folders = get_patient_folders(DATASET_PATH)
if MAX_PATIENTS:
    all_folders = all_folders[:MAX_PATIENTS]

# Exactly match training split (zero-leakage subject-level holdout)
_, _, test_folders = get_subject_splits(all_folders, seed=RANDOM_SEED)
print(f'Test set: {len(test_folders)} patient scans')

## 3️⃣ Run Sliding Window Inference & Compute Metrics

In [ ]:
from src.dataset import load_patient
import nibabel as nib

all_model_results = {}  # model_name → list of per-patient metric dicts

for cfg in MODEL_CONFIGS:
    print(f"\n{'='*60}")
    print(f"Evaluating: {cfg['name']}")
    print(f"{'='*60}")

    patient_results = []

    for folder in tqdm(test_folders, desc=cfg['name']):
        image, gt_seg = load_patient(folder, has_seg=True)
        image_tensor  = torch.from_numpy(image.astype(np.float32))

        # Get voxel spacing from NIfTI header
        from src.dataset import get_file_paths, MODALITIES
        paths = get_file_paths(folder)
        nii   = nib.load(paths['flair'])
        voxel_spacing = tuple(abs(nii.header.get_zooms()[:3]))

        with torch.no_grad():
            pred_seg = sliding_window_inference(
                cfg['model'], image_tensor,
                patch_size=PATCH_SIZE,
                overlap=PATCH_OVERLAP,
                device=DEVICE
            )

        metrics = compute_patient_metrics(pred_seg, gt_seg, voxel_spacing)
        patient_results.append(metrics)

    all_model_results[cfg['name']] = patient_results
    agg = aggregate_metrics(patient_results)
    print_metrics_table(agg, model_name=cfg['name'])

print('\n✅ Evaluation complete for all models')

## 4️⃣ Build Final Comparison Table (Report-Ready)

In [ ]:
rows = []
for cfg in MODEL_CONFIGS:
    agg = aggregate_metrics(all_model_results[cfg['name']])
    row = {'Model': cfg['name']}
    for region in ['WT', 'TC', 'ET']:
        for metric in ['dice', 'iou', 'hd95', 'sensitivity', 'specificity']:
            key = f"{metric.upper()} {region}"
            row[key] = round(agg[region][metric]['mean'], 4)
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df.to_csv('results/comparison_table.csv', index=False)
print('📊 Results saved to results/comparison_table.csv')
print('\n📋 Final Comparison Table:')
print(results_df[['Model','DICE WT','DICE TC','DICE ET','HD95 WT','HD95 ET']].to_string(index=False))

## 5️⃣ Dice Score Comparison Bar Chart

In [ ]:
model_names  = [cfg['name'] for cfg in MODEL_CONFIGS]
short_names  = ['Baseline\nU-Net', 'Channel\nAttn', 'Spatial\nAttn', 'Full Attn\n(Ours)']
regions      = ['WT', 'TC', 'ET']
region_colors = ['#3498db', '#e74c3c', '#2ecc71']

x = np.arange(len(model_names))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 6))

for i, (region, color) in enumerate(zip(regions, region_colors)):
    dice_means = results_df[f'DICE {region}'].values
    bars = ax.bar(x + i * width, dice_means, width, label=f'Dice {region}',
                  color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, dice_means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(short_names, fontsize=11)
ax.set_ylabel('Dice Score', fontsize=12)
ax.set_ylim(0.65, 1.0)
ax.set_title('Dice Score Comparison — All Models & Sub-Regions', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/dice_comparison_barchart.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: results/dice_comparison_barchart.png')

## 6️⃣ Box Plots — Dice Distribution Across Patients

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Dice Score Distribution Across Test Patients', fontsize=13, fontweight='bold')

for ax, region, color in zip(axes, ['WT', 'TC', 'ET'], region_colors):
    data_per_model = []
    for cfg in MODEL_CONFIGS:
        patient_dices = [r[region]['dice'] for r in all_model_results[cfg['name']]]
        data_per_model.append(patient_dices)

    bp = ax.boxplot(data_per_model, patch_artist=True, notch=False,
                    medianprops={'color': 'white', 'linewidth': 2.5})

    colors_box = ['#95a5a6', '#5dade2', '#a9cce3', color]
    for patch, c in zip(bp['boxes'], colors_box):
        patch.set_facecolor(c)
        patch.set_alpha(0.85)

    ax.set_xticks(range(1, len(MODEL_CONFIGS)+1))
    ax.set_xticklabels(['Baseline', 'Ch. Attn', 'Sp. Attn', 'Full\n(Ours)'], fontsize=9)
    ax.set_ylabel('Dice Score')
    ax.set_title(f'{region} — Dice Distribution')
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/dice_boxplots.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: results/dice_boxplots.png')
print('\n🏁 Notebook 05 complete. Proceed to 06_ablation_study.ipynb')